In [ ]:
# parameters
# BINDER_FAST: set flux_pts=30, E_L_pts=20 for fast cloud execution
beta = 0.1       # junction asymmetry ratio
M = 3            # number of large junctions
E_J = 10.0       # large-junction Josephson energy (GHz)
E_C = 0.5        # charging energy (GHz)
E_L = 0.0        # linear inductive energy (GHz); 0 = bare SNAIL
flux_pts = 60    # number of flux points for sweep
E_L_pts = 40     # number of E_L points for participation ratio plot

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.snail import (
    snail_taylor_coefficients,
    snail_circuit_params,
    plot_snail_nonlinearities_vs_flux,
    plot_snail_circuit_params_vs_flux,
)

## Module 2b: SNAIL Circuit Parameters and Flux Tuning

**Learning objectives:**
- Quantise the SNAIL potential and compute $\omega_c$, $\phi_c$, $g_3$, $g_4$, $g_5$, and $\alpha_c$
- Find the Kerr-free flux point where $g_4 = 0$ (or equivalently $\alpha_c = 0$)
- Visualise all circuit parameters over the full flux range
- Understand the role of a geometric inductance $E_L$ (participation ratio $p$)

---

**Sections:** [1 Quantisation](#sec1) · [2 Kerr-Free Point](#sec2) · [3 Full Flux Sweep](#sec3) · [4 Participation Ratio](#sec4)

<a id="sec1"></a>
## 1  Quantisation of the SNAIL Mode

Once we have the Taylor coefficients from the potential, we promote the phase $\varphi$ to a quantum operator and quantise the circuit.

**Oscillator frequency.** The quadratic term $E_J c_2 \varphi^2/2$ together with the charging energy $E_C$ determines the plasma frequency (Chapman *et al.* Eq. B8):

$$\omega_c = \sqrt{8\,\tilde{c}_2\,E_C\,E_J}$$

where $\tilde{c}_j$ are the renormalized Taylor coefficients accounting for an optional geometric inductance (see Section 4). For a bare SNAIL ($E_L = 0$) we have $\tilde{c}_j = c_j$.

**Zero-point phase fluctuation.** The phase operator can be written $\hat{\varphi} = \phi_c(\hat{a} + \hat{a}^\dagger)$, where the zero-point amplitude is:

$$\phi_c = \left(\frac{2E_C}{\tilde{c}_2 E_J}\right)^{1/4}$$

This sets the scale of quantum phase fluctuations in the ground state. A larger $\phi_c$ means the oscillator samples more of the nonlinear part of the potential.

**Nonlinear coupling rates.** Substituting $\hat{\varphi} = \phi_c(\hat{a} + \hat{a}^\dagger)$ into the Taylor-expanded potential:

$$g_3 = \frac{E_J\,\phi_c^3\,\tilde{c}_3}{6}, \quad
  g_4 = \frac{E_J\,\phi_c^4\,\tilde{c}_4}{24}, \quad
  g_5 = \frac{E_J\,\phi_c^5\,\tilde{c}_5}{120}$$

After rotating-wave approximation (RWA), the leading contributions to the self-Kerr anharmonicity are (Chapman *et al.* Eq. 3):

$$\alpha_c = 12\!\left(g_4 - \frac{5\,g_3^2}{\omega_c}\right)$$

The $g_3^2/\omega_c$ term arises from virtual transitions mediated by the cubic interaction — even a pure $g_3$ circuit acquires an effective Kerr from second-order perturbation theory.

*Lab note: Typical values for $M=3$, $\beta=0.1$, $E_J=10\,\text{GHz}$, $E_C=0.5\,\text{GHz}$: $\omega_c/2\pi \approx 4$–$8\,\text{GHz}$, $\phi_c \approx 0.2$–$0.4$, $g_3/2\pi \approx 50$–$150\,\text{MHz}$. The Kerr $\alpha_c$ can be tuned through zero by flux.*

In [ ]:
# Compute circuit parameters at a representative operating point
phi_e_rep = 0.4 * 2 * np.pi
params = snail_circuit_params(beta, M, phi_e_rep, E_J, E_C, E_L=E_L)

print("SNAIL circuit parameters at phi_e = 0.4 * 2pi:")
print(f"  omega_c / 2pi = {params['omega_c']:.4f} GHz")
print(f"  phi_c (ZPF)   = {params['phi_c']:.4f}")
print(f"  g3 / 2pi      = {params['g3']*1e3:.2f} MHz")
print(f"  g4 / 2pi      = {params['g4']*1e3:.2f} MHz")
print(f"  g5 / 2pi      = {params['g5']*1e6:.2f} kHz")
print(f"  alpha_c / 2pi = {params['alpha_c']*1e3:.2f} MHz")
print(f"  xi_crit       = {params['xi_crit']:.4f}")
print(f"  participation p = {params['p']:.4f}")
print()
print("Renormalized Taylor coefficients:")
for key in ['c2t', 'c3t', 'c4t', 'c5t']:
    print(f"  {key} = {params[key]:.6f}")

<a id="sec2"></a>
## 2  The Kerr-Free Point

A powerful feature of the SNAIL is that $\alpha_c$ depends on flux and can be tuned to **zero**. At this operating point the SNAIL is a pure cubic (and quintic) oscillator — it has strong three-wave mixing but no self-Kerr.

**Why Kerr-free matters.** The self-Kerr $\alpha_c$ shifts the oscillator frequency by $\alpha_c$ per photon. In a driven system this creates unwanted photon-number-dependent frequency shifts that de-tune parametric interactions. At the Kerr-free point, the photon-number-dependent shift from $g_4$ is exactly cancelled by the second-order contribution from $g_3$:

$$\alpha_c = 0 \quad \Leftrightarrow \quad g_4 = \frac{5\,g_3^2}{\omega_c}$$

The SNAIL uniquely achieves this because $c_4$ changes sign with flux while $c_3$ remains positive — so there is a flux where the direct quartic and the renormalised-cubic contributions exactly cancel.

*Lab note: The Kerr-free point is the standard operating flux for SNAIL-based parametric amplifiers and beam-splitter experiments. Typical location: $\Phi_e/\Phi_0 \approx 0.41$ for $M=3$, $\beta=0.1$. Slightly off the maximum-$g_3$ point, but the trade-off in $g_3$ is small while the benefit of zero Kerr is large.*

In [ ]:
# Find the Kerr-free point by sweeping flux and interpolating alpha_c = 0
phi_e_scan = np.linspace(0.30 * 2 * np.pi, 0.50 * 2 * np.pi, 200)
alpha_c_scan = np.array([
    snail_circuit_params(beta, M, pe, E_J, E_C, E_L=E_L)['alpha_c']
    for pe in phi_e_scan
])

# Locate sign change and interpolate
sign_changes_kf = np.where(np.diff(np.sign(alpha_c_scan)))[0]
if len(sign_changes_kf) > 0:
    idx_kf = sign_changes_kf[0]
    phi_e_kerr_free = np.interp(
        0,
        [alpha_c_scan[idx_kf], alpha_c_scan[idx_kf + 1]],
        [phi_e_scan[idx_kf], phi_e_scan[idx_kf + 1]],
    )
    print(f"Kerr-free flux: phi_e = {phi_e_kerr_free/(2*np.pi):.4f} * 2pi")
    params_kf = snail_circuit_params(beta, M, phi_e_kerr_free, E_J, E_C, E_L=E_L)
    print(f"  omega_c / 2pi = {params_kf['omega_c']:.4f} GHz")
    print(f"  g3 / 2pi      = {params_kf['g3']*1e3:.2f} MHz")
    print(f"  alpha_c / 2pi = {params_kf['alpha_c']*1e6:.3f} kHz  (should be ~0)")
else:
    print("No Kerr-free point found in scan range — try widening phi_e_scan.")
    phi_e_kerr_free = phi_e_rep  # fallback

In [ ]:
# Plot alpha_c vs flux near the Kerr-free point
fig_kf, ax_kf = plt.subplots(figsize=(6, 4))
ax_kf.plot(phi_e_scan / (2 * np.pi), alpha_c_scan * 1e3,
           lw=1.5, color='C2', label=r"$\alpha_c$")
ax_kf.axhline(0, color='k', lw=0.5, ls='--')
ax_kf.axvline(phi_e_kerr_free / (2 * np.pi), color='crimson', ls=':', lw=1.2,
              label=rf"Kerr-free: $\Phi_e/\Phi_0 = {phi_e_kerr_free/(2*np.pi):.3f}$")
ax_kf.set_xlabel(r"$\Phi_e/\Phi_0$")
ax_kf.set_ylabel(r"$\alpha_c/2\pi$ (MHz)")
ax_kf.set_title("Kerr anharmonicity vs external flux")
ax_kf.legend(fontsize=9, frameon=False)
ax_kf.tick_params(direction='in', top=True, right=True)
plt.tight_layout()
plt.show()

<a id="sec3"></a>
## 3  Full Flux Sweep: Circuit Parameters

Now we compute all six circuit parameters over the full accessible flux range and use the built-in plotting functions.

In [ ]:
# Full flux sweep from 0.05 to 0.5 * 2pi (half period due to symmetry)
phi_e_arr = np.linspace(0.05 * 2 * np.pi, 0.50 * 2 * np.pi, flux_pts)

omega_c_arr = np.zeros(flux_pts)
phi_c_arr   = np.zeros(flux_pts)
g3_arr      = np.zeros(flux_pts)
g4_arr      = np.zeros(flux_pts)
g5_arr      = np.zeros(flux_pts)
alpha_c_arr = np.zeros(flux_pts)
xi_crit_arr = np.zeros(flux_pts)

for i, pe in enumerate(phi_e_arr):
    p = snail_circuit_params(beta, M, pe, E_J, E_C, E_L=E_L)
    omega_c_arr[i] = p['omega_c']
    phi_c_arr[i]   = p['phi_c']
    g3_arr[i]      = p['g3']
    g4_arr[i]      = p['g4']
    g5_arr[i]      = p['g5']
    alpha_c_arr[i] = p['alpha_c']
    xi_crit_arr[i] = p['xi_crit']

print("Flux sweep complete.")
print(f"omega_c range: {omega_c_arr.min():.3f} – {omega_c_arr.max():.3f} GHz")
print(f"g3 range:      {g3_arr.min()*1e3:.1f} – {g3_arr.max()*1e3:.1f} MHz")
print(f"alpha_c range: {alpha_c_arr.min()*1e3:.1f} – {alpha_c_arr.max()*1e3:.1f} MHz")

In [ ]:
# Four-panel circuit parameter plot
fig_cp, axes_cp = plot_snail_circuit_params_vs_flux(
    phi_e_arr, omega_c_arr, phi_c_arr, alpha_c_arr, xi_crit_arr
)
fig_cp.suptitle(
    rf"SNAIL circuit parameters, $\beta={beta}$, $M={M}$, $E_J={E_J}\,\mathrm{{GHz}}$, "
    rf"$E_C={E_C}\,\mathrm{{GHz}}$",
    y=1.02
)
plt.show()

In [ ]:
# Nonlinearities plot
fig_nl, ax_nl = plot_snail_nonlinearities_vs_flux(
    phi_e_arr, g3_arr, g4_arr, g5=g5_arr, units='MHz'
)
fig_nl.suptitle(
    rf"SNAIL nonlinearities, $\beta={beta}$, $M={M}$, $E_J={E_J}\,\mathrm{{GHz}}$, "
    rf"$E_C={E_C}\,\mathrm{{GHz}}$",
    y=1.02
)
plt.show()

*Lab note: The coupler frequency $\omega_c$ decreases monotonically from integer to half-integer flux as the potential flattens. The zero-point phase $\phi_c$ increases correspondingly. The cubic $g_3$ and quintic $g_5$ are both odd under $\phi_e \to 2\pi - \phi_e$ (sign change) while $g_4$ and $\alpha_c$ are even. The critical pump amplitude $\xi_{\rm crit} = 3\pi/(2\phi_c)$ decreases at half-flux (larger $\phi_c$), setting the practical upper bound on pump drive.*

<a id="sec4"></a>
## 4  Participation Ratio and External Inductance

In many experiments the SNAIL is not a bare element but is shunted by a geometric inductance $L_{\rm ext}$ with inductive energy $E_L = (\Phi_0/2\pi)^2/(2L_{\rm ext})$. This occurs in, e.g., coplanar-waveguide SNAIL oscillators where the coupling wire adds linear inductance.

The Josephson inductance *participates* in the total inductive energy with a fraction:

$$p = \frac{c_2 E_J}{E_L + c_2 E_J}$$

For $E_L = 0$ (bare SNAIL), $p = 1$ and all nonlinearities are at their maximum. When $E_L \neq 0$ the renormalized coefficients satisfy $\tilde{c}_n = p^n c_n$ (approximately — see Appendix B of Chapman *et al.* for the exact expressions involving cross-term corrections for $n \geq 4$). The nonlinearities are suppressed as $p^n$, so the quartic and quintic are reduced faster than the cubic.

**Frequency:** $\omega_c = \sqrt{8\,\tilde{c}_2\,E_C\,E_J}$ still holds with the renormalized $\tilde{c}_2$.

*Lab note: In experiments, $E_L$ is typically set by the resonator geometry. The participation ratio $p$ is often extracted by measuring how strongly the coupler frequency is flux-tunable — a coupler with $p \ll 1$ barely moves with flux. Values of $p \approx 0.5$–$0.9$ are common in 3-D cavity SNAIL experiments.*

In [ ]:
# Participation ratio p vs E_L at the representative flux point
E_L_arr = np.linspace(0, 2.0, E_L_pts)
p_arr = np.zeros(E_L_pts)
g3_p_arr = np.zeros(E_L_pts)
omega_c_p_arr = np.zeros(E_L_pts)

for i, el in enumerate(E_L_arr):
    p_params = snail_circuit_params(beta, M, phi_e_kerr_free, E_J, E_C, E_L=el)
    p_arr[i] = p_params['p']
    g3_p_arr[i] = p_params['g3']
    omega_c_p_arr[i] = p_params['omega_c']

fig_p, axes_p = plt.subplots(1, 3, figsize=(11, 3.5))

axes_p[0].plot(E_L_arr, p_arr, lw=1.5, color='C0')
axes_p[0].set_xlabel(r"$E_L$ (GHz)")
axes_p[0].set_ylabel(r"Participation ratio $p$")
axes_p[0].set_title("Participation vs $E_L$")

axes_p[1].plot(E_L_arr, g3_p_arr * 1e3, lw=1.5, color='C1')
axes_p[1].set_xlabel(r"$E_L$ (GHz)")
axes_p[1].set_ylabel(r"$g_3/2\pi$ (MHz)")
axes_p[1].set_title(r"$g_3$ suppression vs $E_L$")

axes_p[2].plot(E_L_arr, omega_c_p_arr, lw=1.5, color='C2')
axes_p[2].set_xlabel(r"$E_L$ (GHz)")
axes_p[2].set_ylabel(r"$\omega_c/2\pi$ (GHz)")
axes_p[2].set_title(r"$\omega_c$ vs $E_L$")

for ax in axes_p:
    ax.tick_params(direction='in', top=True, right=True)

fig_p.suptitle(
    rf"Effect of geometric inductance, $\phi_e = {phi_e_kerr_free/(2*np.pi):.3f}\times 2\pi$ (Kerr-free)",
    y=1.02
)
plt.tight_layout()
plt.show()

print(f"At E_L = 0:    p = {p_arr[0]:.4f},  g3 = {g3_p_arr[0]*1e3:.2f} MHz")
print(f"At E_L = 1.0:  p = {p_arr[np.argmin(np.abs(E_L_arr - 1.0))]:.4f},  "
      f"g3 = {g3_p_arr[np.argmin(np.abs(E_L_arr - 1.0))]*1e3:.2f} MHz")